# Semana 8 · Sesión 1: Lagrangianos y `LagrangesMethod`

**Módulo 2**

## Objetivos de la sesión

1. Construir la energía cinética y la potencial de un sistema con `Particle`, y
   armar su Lagrangiano con `me.Lagrangian`.
2. Generar las ecuaciones de Euler-Lagrange con `LagrangesMethod` y leerlas
   físicamente.
3. Escribir las ecuaciones en la forma $M\ddot{\mathbf{q}} = \mathbf{f}$, y
   linealizarlas alrededor del equilibrio para obtener los modos normales.

## Retomamos

La semana pasada hicimos **cinemática**: marcos, puntos, velocidades y
aceleraciones. Sabemos *cómo* se mueve algo si alguien nos dice cómo cambian
sus ángulos, pero no *por qué* se mueve así. Al péndulo doble le **impusimos**
un movimiento.

Esta semana pasamos a la **dinámica**: a partir de las masas, las fuerzas y la
cinemática, `mechanics` va a generar las **ecuaciones de movimiento** por
nosotros. Hoy con la formulación de Lagrange; la próxima sesión, con la de
Kane, y las vamos a integrar numéricamente.

La celda de abajo prepara lo mismo que la semana pasada: el tiempo, el marco
del laboratorio y un origen fijo.

In [ ]:
import sympy as sp
import sympy.physics.mechanics as me
from IPython.display import Math

me.init_vprinting()

t = sp.Symbol("t")               # el tiempo de mechanics: sin suposiciones
N = me.ReferenceFrame("N")       # el marco del laboratorio
O = me.Point("O")
O.set_vel(N, 0)                  # el pivote, fijo en el laboratorio

## El Lagrangiano, a mano

Recordatorio del curso de mecánica: para un sistema con coordenadas
generalizadas $q_i$, el **Lagrangiano** es la energía cinética menos la
potencial, $\mathcal{L} = T - V$, y las ecuaciones de movimiento son las de
**Euler-Lagrange**, una por coordenada:

$$\frac{d}{dt}\frac{\partial \mathcal{L}}{\partial \dot q_i}
  - \frac{\partial \mathcal{L}}{\partial q_i} = 0$$

Empezamos con el sistema más sencillo, un péndulo simple de longitud $L$ y
masa $m$, y lo hacemos **a mano**, con `diff` como en la semana 5. La
velocidad de la masa es $L\dot\theta$ y su altura respecto al pivote es
$-L\cos\theta$, así que

$$T = \tfrac12 m L^2 \dot\theta^2, \qquad V = -m g L \cos\theta .$$

In [ ]:
theta = me.dynamicsymbols("theta")
m, g, L = sp.symbols("m g L", positive=True)   # masa, gravedad y longitud

T = sp.Rational(1, 2) * m * L**2 * theta.diff(t)**2
V = -m * g * L * sp.cos(theta)

lagrangiano_mano = T - V
display(sp.Eq(sp.Symbol(r"\mathcal{L}"), lagrangiano_mano))

In [ ]:
# d/dt (dL/d theta_punto) - dL/d theta
euler_lagrange = (
    sp.diff(sp.diff(lagrangiano_mano, theta.diff(t)), t)
    - sp.diff(lagrangiano_mano, theta)
)
display(sp.Eq(euler_lagrange, 0))

Dividiendo entre $mL^2$ sale la ecuación de siempre,
$\ddot\theta = -\frac{g}{L}\sin\theta$.

Fíjate en qué partes fueron trabajo **nuestro** y no de SymPy: escribir $T$
—para lo que había que saber la velocidad— y escribir $V$ —para lo que había
que saber la altura—. Con un péndulo simple es fácil. Con un péndulo doble, la
velocidad de la segunda masa ya no se escribe de memoria: por eso la semana
pasada aprendimos a pedírsela a `mechanics`. Ahora juntamos las dos cosas.

## `Particle`: una masa en un punto

Una **partícula** es una masa concentrada en un `Point`. Como el punto ya sabe
su velocidad, la partícula sabe su energía cinética.

| Quieres | Escribe |
|---|---|
| Crear una partícula | `me.Particle("nombre", punto, masa)` |
| Su energía cinética en un marco | `particula.kinetic_energy(N)` |
| Asignarle energía potencial | `particula.potential_energy = expresion` |
| El Lagrangiano de varias | `me.Lagrangian(N, p1, p2, ...)` |

La energía cinética se pide **en un marco**, como la velocidad. La potencial,
en cambio, no la puede adivinar: ¿hay gravedad?, ¿resortes?, ¿un campo
eléctrico? Esa se la damos nosotros.

In [ ]:
A = me.ReferenceFrame("A")
A.orient_axis(N, N.z, theta)     # la varilla gira con theta

P = O.locatenew("P", -L * A.y)   # la masa cuelga a una distancia L del pivote
bola = me.Particle("bola", P, m)

display(sp.Eq(sp.Symbol("T"), bola.kinetic_energy(N)))

Para la potencial gravitacional usamos la altura: la componente vertical de la
posición, $\mathbf{r}_{P/O} \cdot \hat{n}_y$. Así no tenemos que pensar el
coseno: sale del producto punto.

In [ ]:
altura = P.pos_from(O).dot(N.y)
bola.potential_energy = m * g * altura

lagrangiano = me.Lagrangian(N, bola)
display(sp.Eq(sp.Symbol(r"\mathcal{L}"), lagrangiano))

# Verificar en lugar de creer: ¿es el mismo que el de a mano?
display(sp.Eq(sp.Symbol(r"\mathcal{L} - \mathcal{L}_{\mathrm{mano}}"),
              sp.simplify(lagrangiano - lagrangiano_mano)))

## `LagrangesMethod`: las ecuaciones de movimiento

Con el Lagrangiano y la lista de coordenadas generalizadas, `LagrangesMethod`
hace las derivadas de Euler-Lagrange.

| Paso | Escribe | Devuelve |
|---|---|---|
| Plantear | `metodo = me.LagrangesMethod(lagrangiano, [q1, q2, ...])` | El objeto, sin calcular nada aún |
| Generar las ecuaciones | `metodo.form_lagranges_equations()` | Una `sp.Matrix` columna, un renglón por coordenada |

Cada renglón es una expresión que **vale cero**: la ecuación de Euler-Lagrange
de esa coordenada. Por eso la mostramos con `sp.Eq(..., 0)`.

In [ ]:
metodo = me.LagrangesMethod(lagrangiano, [theta])
ecuaciones = metodo.form_lagranges_equations()

display(sp.Eq(ecuaciones[0], 0))

La misma ecuación que a mano. El orden importa: primero
`form_lagranges_equations()`, y solo después se le pueden pedir cosas al
método. Si se nos olvida, el error lo dice claro.

In [ ]:
sin_ecuaciones = me.LagrangesMethod(lagrangiano, [theta])

try:
    sin_ecuaciones.rhs()
except ValueError as error:
    print("ValueError -", error)

## TODO en clase 1

Un **péndulo elástico**: la masa $m$ cuelga del pivote de un resorte de
constante $k$ y longitud natural $\ell_0$, en lugar de una varilla rígida.
Ahora hay **dos** coordenadas: la longitud del resorte $r(t)$ y el ángulo
$\theta(t)$.

![Péndulo elástico: una masa m cuelga del pivote O de un resorte de constante k; su longitud r y su ángulo θ respecto a la vertical son las dos coordenadas](img/pendulo-elastico.svg)

1. Declara `k` y `l_0` (nombres de símbolo `"k"` y `"l_0"`), positivos, y `r`
   como `dynamicsymbol`. Puedes reusar el marco `A`: gira con el mismo
   $\theta$.
2. Ubica el punto `Q` a una distancia $r$ del pivote, a lo largo de $-\hat{a}_y$,
   y crea la partícula `masa_elastica`.
3. Su energía potencial tiene **dos** términos: la gravitacional, como arriba, y
   la del resorte, $\tfrac12 k (r - \ell_0)^2$.
4. Arma `lagrangiano_elastico` y `ecuaciones_elasticas` con `LagrangesMethod`,
   con las coordenadas en el orden `[r, theta]`, y muestra cada una con
   `sp.simplify`. Deben salir, reacomodadas,

   $$m\ddot r - m r\dot\theta^2 + k(r - \ell_0) - m g\cos\theta = 0$$

   $$m r^2\ddot\theta + 2 m r\dot r\dot\theta + m g r\sin\theta = 0$$

5. Busca en ellas los términos de la semana pasada: ¿cuál es el centrípeto y
   cuál el de Coriolis?

In [ ]:
# TODO en clase: el péndulo elástico
k = ...
l_0 = ...
r = ...

Q = ...
masa_elastica = ...

lagrangiano_elastico = ...
ecuaciones_elasticas = ...

## Física: el péndulo doble

Volvemos al sistema de la semana pasada: una varilla de longitud $L_1$ con una
masa $m_1$ en la punta, colgando del pivote con ángulo $\theta_1$, y de ella
otra de longitud $L_2$ y masa $m_2$, con ángulo $\theta_2$. Los dos ángulos se
miden respecto a la vertical.

![Péndulo doble: la varilla L1 cuelga de O con ángulo θ1 y de su extremo P1 cuelga la varilla L2 con ángulo θ2, ambos medidos desde la vertical](img/pendulo-doble.svg)

La cinemática es la de la semana 7, resuelta. Lo nuevo son las partículas y
sus energías potenciales.

In [ ]:
theta1, theta2 = me.dynamicsymbols("theta1 theta2")
m1, m2, L1, L2 = sp.symbols("m1 m2 L1 L2", positive=True)

A1 = me.ReferenceFrame("A1")
A1.orient_axis(N, N.z, theta1)
A2 = me.ReferenceFrame("A2")
A2.orient_axis(N, N.z, theta2)

P1 = O.locatenew("P1", -L1 * A1.y)
P2 = P1.locatenew("P2", -L2 * A2.y)

particula_1 = me.Particle("particula_1", P1, m1)
particula_2 = me.Particle("particula_2", P2, m2)
particula_1.potential_energy = m1 * g * P1.pos_from(O).dot(N.y)
particula_2.potential_energy = m2 * g * P2.pos_from(O).dot(N.y)

In [ ]:
lagrangiano_doble = me.Lagrangian(N, particula_1, particula_2)
display(sp.Eq(sp.Symbol(r"\mathcal{L}"), sp.trigsimp(sp.expand(lagrangiano_doble))))

Ahí está el término $L_1 L_2 m_2 \dot\theta_1\dot\theta_2\cos(\theta_1 - \theta_2)$:
es el que salió en el `TODO en clase 2` de la semana pasada, en la rapidez al
cuadrado de la segunda masa. Es el que **acopla** los dos péndulos: sin él,
serían dos péndulos simples independientes.

Las ecuaciones de movimiento son el paso que a mano cuesta media hoja por
ecuación.

In [ ]:
metodo_doble = me.LagrangesMethod(lagrangiano_doble, [theta1, theta2])
ecuaciones_doble = metodo_doble.form_lagranges_equations()

display(sp.Eq(sp.simplify(ecuaciones_doble[0]), 0))
display(sp.Eq(sp.simplify(ecuaciones_doble[1]), 0))

## La forma $M\ddot{\mathbf{q}} = \mathbf{f}$

Las dos ecuaciones mezclan $\ddot\theta_1$ y $\ddot\theta_2$, pero siempre de
forma **lineal**: las aceleraciones nunca aparecen al cuadrado ni dentro de un
seno. Eso permite escribirlas como un sistema matricial,

$$M(\mathbf{q})\,\ddot{\mathbf{q}} = \mathbf{f}(\mathbf{q}, \dot{\mathbf{q}}, t)$$

con $M$ la **matriz de masa** y $\mathbf{f}$ el **vector de fuerzas**, que
reúne todo lo que no es aceleración. `LagrangesMethod` ya los tiene separados:

| Atributo | Contiene |
|---|---|
| `metodo.mass_matrix` | $M$ |
| `metodo.forcing` | $\mathbf{f}$ |

In [ ]:
display(sp.Eq(sp.Symbol("M"), sp.simplify(metodo_doble.mass_matrix), evaluate=False))
display(sp.Eq(sp.Symbol(r"\mathbf{f}"), sp.simplify(metodo_doble.forcing), evaluate=False))

$M$ es simétrica, como la matriz de masas de la semana 6, pero ahora **depende
de la configuración**: las entradas de fuera de la diagonal llevan
$\cos(\theta_1 - \theta_2)$.

## Primer orden: lo que pide un integrador

Los integradores numéricos, como el que usaremos la próxima sesión, no saben
resolver ecuaciones de segundo orden. Piden un sistema de **primer orden**
$\dot{\mathbf{x}} = F(\mathbf{x}, t)$, con un vector de estado que junta
posiciones y velocidades, $\mathbf{x} = (\mathbf{q}, \dot{\mathbf{q}})$.

`metodo.rhs()` —*right-hand side*, el lado derecho— hace ese cambio y despeja
las aceleraciones. Para el péndulo simple:

In [ ]:
display(sp.Eq(sp.Symbol(r"\dot{\mathbf{x}}"), metodo.rhs(), evaluate=False))

El primer renglón solo dice que la derivada de $\theta$ es $\dot\theta$; el
segundo es la física. Para el péndulo doble `rhs()` funciona igual, pero
despejar requiere invertir $M$ y la expresión crece mucho: no vale la pena
imprimirla. Basta con saber que existe, porque es justo lo que vamos a darle
al integrador.

## Fuerzas que no vienen de un potencial

$\mathcal{L} = T - V$ solo sabe de fuerzas **conservativas**. La fricción no
tiene energía potencial. `LagrangesMethod` la acepta aparte, en una lista de
pares `(punto, fuerza)`, junto con el marco en el que se calculan las
velocidades:

```python
me.LagrangesMethod(lagrangiano, [q], forcelist=[(punto, fuerza)], frame=N)
```

Con una fricción viscosa $-b\,\mathbf{v}$ sobre la masa del péndulo simple:

In [ ]:
b = sp.Symbol("b", positive=True)   # coeficiente de fricción viscosa

friccion = -b * P.vel(N)
metodo_friccion = me.LagrangesMethod(lagrangiano, [theta],
                                     forcelist=[(P, friccion)], frame=N)

display(sp.Eq(metodo_friccion.form_lagranges_equations()[0], 0))

Apareció $bL^2\dot\theta$: la fricción frena. Funciona, pero el Lagrangiano ya
no cuenta toda la historia —hay que llevar las fuerzas por fuera—. La
formulación de Kane, la próxima sesión, trata a todas las fuerzas por igual
desde el principio.

## TODO en clase 2

¿Cómo oscila el péndulo doble cuando los ángulos son **pequeños**? Cerca del
equilibrio $\theta_1 = \theta_2 = 0$ (y velocidades cero), las ecuaciones
se vuelven lineales y toman la forma de la semana 6,
$M_0\,\ddot{\mathbf{q}} = -K\,\mathbf{q}$, donde:

- $M_0$ es la matriz de masa evaluada en el equilibrio.
- $K$ es la matriz de rigidez: menos el jacobiano de $\mathbf{f}$ respecto a
  $(\theta_1, \theta_2)$, evaluado en el equilibrio. Es la versión matricial
  de $\sin\theta \approx \theta$.

1. Arma el diccionario `equilibrio` que manda $\theta_1$, $\theta_2$,
   $\dot\theta_1$ y $\dot\theta_2$ a cero.
2. Calcula `matriz_masa` $= M_0$ y `rigidez` $= K$ (con `.jacobian([theta1, theta2])`
   y `.subs(equilibrio)`). Muéstralas: ¿son simétricas?
3. Las frecuencias al cuadrado de los modos normales son los eigenvalores de
   $M_0^{-1}K$. Calcula `frecuencias_cuadradas` para el caso de masas y
   longitudes iguales, sustituyendo `{m1: m, m2: m, L1: L, L2: L}`. Deben
   salir
   $$\omega^2 = (2 \pm \sqrt{2})\,\frac{g}{L} .$$
4. Obtén los eigenvectores. ¿Cómo se mueven los dos péndulos en cada modo: en
   fase o a contrafase? ¿Cuál es más rápido?

In [ ]:
# TODO en clase: modos normales del péndulo doble
equilibrio = ...

matriz_masa = ...
rigidez = ...

frecuencias_cuadradas = ...

## Resumen

Hoy pasamos de la cinemática a la dinámica. Una `Particle` pone una masa en un
punto: su energía cinética sale de la velocidad del punto, y la potencial se la
damos nosotros. `me.Lagrangian` arma $T - V$, y `LagrangesMethod` hace las
derivadas de Euler-Lagrange, una ecuación por coordenada. Lo comprobamos a mano
con el péndulo simple, y lo usamos donde a mano ya duele: el péndulo elástico y
el péndulo doble.

Las ecuaciones tienen la forma $M\ddot{\mathbf{q}} = \mathbf{f}$; `rhs()` las
pasa a primer orden, que es lo que pide un integrador. Cerca del equilibrio se
linealizan en $M_0\ddot{\mathbf{q}} = -K\mathbf{q}$, y los modos normales son,
otra vez, un problema de eigenvalores.

**Siguiente sesión:** la formulación de **Kane** con `KanesMethod` —otra ruta a
las mismas ecuaciones, más natural con fricción y con muchos cuerpos— y, por
fin, **integrar** las ecuaciones del péndulo doble para verlo moverse como la
física manda.